# Paper Compilation: Recompute Statistics from Raw Data

**Evaluation artifact** that loads experiment results across 14 datasets (8 real + 6 synthetic),
recomputes statistical tests (Friedman, Wilcoxon, Hedges' g, Spearman), generates matplotlib
figures, and builds results tables for a complete paper on
*Spectral Decomposition of Co-Information Feature Graphs for Interpretable Oblique Decision Trees*.

This demo loads pre-computed per-method evaluation results and recreates the statistical
analysis pipeline: Friedman test across 8 methods, arity comparison via Wilcoxon signed-rank,
signed vs. unsigned ablation via Hedges' g, and frustration correlation via Spearman rho.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# scikit-posthocs — NOT on Colab, always install
_pip('scikit-posthocs==0.11.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
# scipy==1.16.3 requires Python>=3.11; use 1.15.3 for Python 3.10 compat
if 'google.colab' not in sys.modules:
    _scipy = 'scipy==1.16.3' if sys.version_info >= (3, 11) else 'scipy==1.15.3'
    _pip('numpy==2.0.2', 'pandas==2.2.2', _scipy, 'matplotlib==3.10.0', 'tabulate==0.9.0')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import os
import math
import numpy as np
import pandas as pd
from scipy import stats
import scikit_posthocs as sp

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/evaluation_iter7_paper_compilati/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data: {len(data['datasets'])} datasets, "
      f"{sum(len(d['examples']) for d in data['datasets'])} examples")
print(f"Paper: {data['metadata']['paper_title']}")

Loaded data: 14 datasets, 94 examples
Paper: Spectral Decomposition of Co-Information Feature Graphs for Interpretable Oblique Decision Trees


In [5]:
# ── Configuration ──
# Dataset and method definitions (from original eval.py)
CLF_DATASETS = ["adult", "electricity", "jannis", "higgs_small", "eye_movements", "credit", "miniboone"]
REG_DATASETS = ["california_housing"]
ALL_REAL_DATASETS = CLF_DATASETS + REG_DATASETS
SYNTHETIC_VARIANTS = ["easy_2mod_xor", "medium_4mod_mixed", "overlapping_modules",
                      "no_structure_control", "hard_4mod_unequal", "highdim_8mod"]

FIGS_METHODS = ["axis_aligned", "random_oblique", "unsigned_spectral", "signed_spectral", "hard_threshold"]
BASELINE_METHODS = ["ebm", "random_forest", "linear"]
ALL_METHODS = FIGS_METHODS + BASELINE_METHODS
CLUSTERING_METHODS = ["unsigned_spectral", "signed_spectral", "hard_threshold"]

METHOD_DISPLAY = {
    "axis_aligned": "AA-FIGS", "random_oblique": "RO-FIGS",
    "unsigned_spectral": "US-FIGS", "signed_spectral": "SS-FIGS",
    "hard_threshold": "HT-FIGS", "ebm": "EBM",
    "random_forest": "RF", "linear": "Linear",
}
DS_DISPLAY = {
    "adult": "Adult", "electricity": "Elec.", "jannis": "Jannis",
    "higgs_small": "Higgs", "eye_movements": "Eye", "credit": "Credit",
    "miniboone": "MiniBoo.", "california_housing": "CalHous.",
}

# Tunable: number of datasets to process (min=2 for stats, full=7 clf + 1 reg)
# Original values used in paper
N_CLF_DATASETS = len(CLF_DATASETS)  # original: 7
N_METHODS = len(ALL_METHODS)  # original: 8

## Step 1: Extract Per-Dataset, Per-Method Results

Parse the loaded evaluation data into a table structure: for each (dataset, method) pair,
extract balanced accuracy mean/std and split arity.

In [6]:
# Build table1: {dataset: {method: {"mean": ..., "std": ...}}}
# and arity_map: {(dataset, method): arity_mean}
table1 = {}
arity_map = {}

for ds_entry in data["datasets"]:
    ds = ds_entry["dataset"]
    if ds not in ALL_REAL_DATASETS:
        continue
    table1[ds] = {}
    for ex in ds_entry["examples"]:
        meth = ex["metadata_method"]
        out = json.loads(ex["output"])
        bacc_mean = out.get("balanced_accuracy_mean", 0.0)
        bacc_std = out.get("balanced_accuracy_std", 0.0)
        arity = out.get("avg_split_arity_mean", 1.0)
        table1[ds][meth] = {"mean": bacc_mean, "std": bacc_std}
        arity_map[(ds, meth)] = arity

# Build synthetic results
synthetic_acc = {}
module_recovery = {}
for ds_entry in data["datasets"]:
    ds = ds_entry["dataset"]
    if ds not in SYNTHETIC_VARIANTS:
        continue
    for ex in ds_entry["examples"]:
        meth = ex["metadata_method"]
        out = json.loads(ex["output"])
        synthetic_acc[(ds, meth)] = {
            "bacc_mean": out.get("balanced_accuracy_mean", 0.0),
            "bacc_std": out.get("balanced_accuracy_std", 0.0),
        }
        if meth in CLUSTERING_METHODS:
            module_recovery[(ds, meth)] = {
                "ari_mean": out.get("module_recovery_ari", 0.0),
                "jaccard_mean": out.get("module_recovery_jaccard", 0.0),
            }

print(f"Table 1 built: {len(table1)} datasets x {len(ALL_METHODS)} methods")
print(f"Arity entries: {len(arity_map)}")
print(f"Synthetic accuracy entries: {len(synthetic_acc)}")
print(f"Module recovery entries: {len(module_recovery)}")

Table 1 built: 8 datasets x 8 methods
Arity entries: 64
Synthetic accuracy entries: 30
Module recovery entries: 18


## Step 2: Friedman Test + Nemenyi Post-Hoc

Test whether the 8 methods differ significantly across 7 classification datasets using the
non-parametric Friedman test, followed by Nemenyi post-hoc pairwise comparisons.

In [7]:
# Step 2C: Friedman test + Nemenyi post-hoc (7 clf datasets, 8 methods)
matrix = []
for ds in CLF_DATASETS[:N_CLF_DATASETS]:
    row = []
    for meth in ALL_METHODS[:N_METHODS]:
        val = table1.get(ds, {}).get(meth, {}).get("mean")
        row.append(val if val is not None else 0.5)
    matrix.append(row)
matrix = np.array(matrix)
print(f"Friedman matrix shape: {matrix.shape}")

try:
    chi2, p = stats.friedmanchisquare(*matrix.T)
    print(f"Friedman chi2={chi2:.4f}, p={p:.6f}")
except Exception as e:
    print(f"Friedman test failed: {e}")
    chi2, p = float("nan"), float("nan")

nemenyi_df = None
try:
    nemenyi_df = sp.posthoc_nemenyi_friedman(matrix)
    nemenyi_df.index = ALL_METHODS[:N_METHODS]
    nemenyi_df.columns = ALL_METHODS[:N_METHODS]
    print("Nemenyi post-hoc computed")
    print(nemenyi_df.round(4).to_string())
except Exception as e:
    print(f"Nemenyi failed: {e}")

Friedman matrix shape: (7, 8)
Friedman chi2=37.0952, p=0.000004


Nemenyi post-hoc computed
                   axis_aligned  random_oblique  unsigned_spectral  signed_spectral  hard_threshold     ebm  random_forest  linear
axis_aligned             1.0000          0.9981             0.8957           0.8957          0.9589  0.0113         0.0113  0.8957
random_oblique           0.9981          1.0000             0.9981           0.9981          0.9999  0.0861         0.0861  0.5065
unsigned_spectral        0.8957          0.9981             1.0000           1.0000          1.0000  0.3625         0.3625  0.1491
signed_spectral          0.8957          0.9981             1.0000           1.0000          1.0000  0.3625         0.3625  0.1491
hard_threshold           0.9589          0.9999             1.0000           1.0000          1.0000  0.2409         0.2409  0.2409
ebm                      0.0113          0.0861             0.3625           0.3625          0.2409  1.0000         1.0000  0.0000
random_forest            0.0113          0.0861          

## Step 3: Arity Analysis — Wilcoxon Signed-Rank Test

Compare split arity between Unsigned Spectral and Random Oblique FIGS methods.
Lower arity = fewer features per split = more interpretable models.

In [8]:
# Step 2D: Wilcoxon signed-rank test for arity (unsigned_spectral vs random_oblique)
arity_u, arity_r = [], []
for ds in CLF_DATASETS[:N_CLF_DATASETS]:
    u = arity_map.get((ds, "unsigned_spectral"), 1.0)
    r = arity_map.get((ds, "random_oblique"), 1.0)
    arity_u.append(u)
    arity_r.append(r)

arity_u = np.array(arity_u)
arity_r = np.array(arity_r)
diff = arity_r - arity_u

try:
    W, wilcoxon_p = stats.wilcoxon(arity_r, arity_u, alternative="two-sided")
    d = float(np.mean(diff) / np.std(diff, ddof=1)) if np.std(diff, ddof=1) > 0 else 0.0
    n = len(diff)
    g_corr = 1 - 3 / (4 * (2 * n) - 9)
    hedges_g_arity = d * g_corr
    print(f"Wilcoxon W={W:.4f}, p={wilcoxon_p:.6f}")
    print(f"Cohen's d={d:.4f}, Hedges' g={hedges_g_arity:.4f}")
except Exception as e:
    print(f"Wilcoxon test failed: {e}")
    W, wilcoxon_p, d, hedges_g_arity = float("nan"), float("nan"), float("nan"), float("nan")

# Step 2F: Interpretability metrics (arity, path length, cognitive complexity)
interp = {}
for meth in FIGS_METHODS:
    arities = []
    for ds in CLF_DATASETS[:N_CLF_DATASETS]:
        a = arity_map.get((ds, meth), 1.0)
        arities.append(a)
    ma = float(np.mean(arities)) if arities else 1.0
    interp[meth] = {"mean_arity": ma}
    print(f"  {METHOD_DISPLAY[meth]}: mean_arity={ma:.3f}")

Wilcoxon W=6.0000, p=0.218750
Cohen's d=-0.6652, Hedges' g=-0.6227
  AA-FIGS: mean_arity=1.000
  RO-FIGS: mean_arity=3.244
  US-FIGS: mean_arity=10.937
  SS-FIGS: mean_arity=6.641
  HT-FIGS: mean_arity=9.923


## Step 4: Hedges' g — Signed vs. Unsigned Spectral Ablation

Compute effect size (Hedges' g) between unsigned and signed spectral FIGS
across classification datasets to quantify the ablation.

In [9]:
# Step 2E: Hedges' g for signed vs unsigned ablation
# Using means from table1 (we don't have per-fold data, so we use dataset-level means)
hedges_g_per_ds = {}
unsigned_means, signed_means = [], []
for ds in CLF_DATASETS[:N_CLF_DATASETS]:
    u_mean = table1.get(ds, {}).get("unsigned_spectral", {}).get("mean")
    s_mean = table1.get(ds, {}).get("signed_spectral", {}).get("mean")
    u_std = table1.get(ds, {}).get("unsigned_spectral", {}).get("std", 0.01)
    s_std = table1.get(ds, {}).get("signed_spectral", {}).get("std", 0.01)
    if u_mean is not None and s_mean is not None:
        # Approximate Hedges' g from means and stds
        sp2 = (u_std**2 + s_std**2) / 2
        s_pooled = np.sqrt(sp2) if sp2 > 0 else 1e-12
        g = float((u_mean - s_mean) / s_pooled)
        hedges_g_per_ds[ds] = g
        unsigned_means.append(u_mean)
        signed_means.append(s_mean)
        print(f"  {DS_DISPLAY.get(ds, ds)}: g={g:.4f} (US={u_mean:.4f}, SS={s_mean:.4f})")

# Pooled Hedges' g from metadata
pooled_hedges_g = data["metadata"].get("signed_vs_unsigned_pooled_hedges_g", 0.0)
print(f"\nPooled Hedges' g (from full per-fold analysis): {pooled_hedges_g:.6f}")

  Adult: g=0.0235 (US=0.7026, SS=0.7024)
  Elec.: g=-0.8170 (US=0.7912, SS=0.7943)
  Jannis: g=-1.0009 (US=0.7342, SS=0.7414)
  Higgs: g=1.7828 (US=0.6756, SS=0.6714)
  Eye: g=-0.5604 (US=0.5727, SS=0.5815)
  Credit: g=0.0672 (US=0.7676, SS=0.7673)
  MiniBoo.: g=3.8037 (US=0.9018, SS=0.8917)

Pooled Hedges' g (from full per-fold analysis): -0.006189


## Step 5: Frustration Correlation (Spearman)

Test whether the graph frustration index predicts the benefit of oblique splits
across all 14 datasets (8 real + 6 synthetic).

In [10]:
# Step 2H: Frustration correlation (recompute from supplementary data)
frust_data = data.get("supplementary", {}).get("frustration_data", [])
frustrations = [d["frustration_index"] for d in frust_data]
benefits = [d["oblique_benefit"] for d in frust_data]
labels = [d["dataset"] for d in frust_data]
is_synth = [d["is_synthetic"] for d in frust_data]

if len(frustrations) >= 3:
    spearman_rho, spearman_p = stats.spearmanr(frustrations, benefits)
else:
    spearman_rho, spearman_p = float("nan"), float("nan")

print(f"Spearman rho={spearman_rho:.4f}, p={spearman_p:.4f}, n={len(frustrations)}")
print(f"Reference from metadata: rho={data['metadata']['frustration_spearman_rho']:.4f}, "
      f"p={data['metadata']['frustration_spearman_p']:.4f}")

Spearman rho=-0.2659, p=0.3581, n=14
Reference from metadata: rho=-0.1077, p=0.7140


## Step 6: Generate Figures

Recreate the 4 paper figures:
1. **Pipeline diagram** — method overview
2. **Arity comparison** — bar chart across datasets
3. **Frustration scatter** — frustration index vs oblique benefit
4. **CoI sign distribution** — stacked bars per dataset

In [11]:
# Figure 1: Pipeline diagram
fig, ax = plt.subplots(1, 1, figsize=(14, 2.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 2.5)
ax.axis("off")

boxes = [
    (0.2, "Raw\nFeatures"),
    (2.2, "Pairwise\nCoI Matrix"),
    (4.2, "|CoI|\nGraph"),
    (6.2, "Spectral\nClustering"),
    (8.2, "Feature\nModules"),
    (10.2, "Module-Constrained\nOblique FIGS"),
    (12.2, "Interpretable\nTrees"),
]
colors = ["#E8F4FD", "#D1ECF1", "#BEE5EB", "#A2D9CE", "#82E0AA", "#F9E79F", "#FADBD8"]

for i, (x, label) in enumerate(boxes):
    rect = FancyBboxPatch((x, 0.5), 1.6, 1.4, boxstyle="round,pad=0.1",
                          facecolor=colors[i], edgecolor="#2C3E50", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.8, 1.2, label, ha="center", va="center", fontsize=8,
            fontweight="bold", color="#2C3E50")
    if i < len(boxes) - 1:
        ax.annotate("", xy=(boxes[i + 1][0] - 0.05, 1.2),
                    xytext=(x + 1.65, 1.2),
                    arrowprops=dict(arrowstyle="->", color="#2C3E50", lw=1.5))

plt.title("Figure 1: Spectral CoI Feature Module Pipeline", fontsize=12, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()
plt.close(fig)

# Figure 2: Arity comparison bar chart
methods_to_plot = ["axis_aligned", "random_oblique", "unsigned_spectral"]
colors_m = {"axis_aligned": "#95A5A6", "random_oblique": "#E74C3C", "unsigned_spectral": "#3498DB"}
labels_m = {"axis_aligned": "Axis-Aligned", "random_oblique": "Random Oblique", "unsigned_spectral": "Unsigned Spectral"}

x = np.arange(len(CLF_DATASETS[:N_CLF_DATASETS]))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4))

for i, meth in enumerate(methods_to_plot):
    means = []
    for ds in CLF_DATASETS[:N_CLF_DATASETS]:
        means.append(arity_map.get((ds, meth), 1.0))
    ax.bar(x + i * width, means, width, label=labels_m[meth],
           color=colors_m[meth], capsize=3, edgecolor="white", linewidth=0.5)

ax.set_xlabel("Dataset", fontsize=11)
ax.set_ylabel("Mean Split Arity", fontsize=11)
ax.set_title("Figure 2: Split Arity Comparison Across Classification Datasets", fontsize=12, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels([DS_DISPLAY.get(d, d) for d in CLF_DATASETS[:N_CLF_DATASETS]], rotation=30, ha="right")
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

# Figure 3: Frustration vs oblique benefit scatter
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(len(frustrations)):
    marker = "^" if is_synth[i] else "o"
    color = "#E74C3C" if is_synth[i] else "#3498DB"
    ax.scatter(frustrations[i], benefits[i], marker=marker, color=color, s=80, zorder=5,
               edgecolors="white", linewidth=0.5)
    ax.annotate(labels[i], (frustrations[i], benefits[i]), fontsize=6.5,
                textcoords="offset points", xytext=(5, 5), alpha=0.8)

if len(frustrations) >= 3:
    f_arr, b_arr = np.array(frustrations), np.array(benefits)
    slope, intercept, _, _, _ = stats.linregress(f_arr, b_arr)
    x_line = np.linspace(min(f_arr), max(f_arr), 100)
    ax.plot(x_line, slope * x_line + intercept, "--", color="#7F8C8D", alpha=0.7, linewidth=1.5)
    rho_v, p_v = stats.spearmanr(frustrations, benefits)
    ax.text(0.05, 0.95, f"Spearman rho={rho_v:.3f}, p={p_v:.3f}",
            transform=ax.transAxes, fontsize=10, va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))

ax.set_xlabel("Frustration Index (normalized)", fontsize=11)
ax.set_ylabel("Oblique Benefit (balanced accuracy)", fontsize=11)
ax.set_title("Figure 3: Frustration Index vs. Oblique Benefit Across 14 Datasets", fontsize=12, fontweight="bold")
real_patch = plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#3498DB", markersize=8, label="Real")
synth_patch = plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="#E74C3C", markersize=8, label="Synthetic")
ax.legend(handles=[real_patch, synth_patch], fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

# Figure 4: CoI sign distribution stacked bar chart
sign_dist = data.get("supplementary", {}).get("sign_distribution", {})
datasets_sd = [ds for ds in ALL_REAL_DATASETS if ds in sign_dist]
pos = [sign_dist[ds]["frac_positive"] for ds in datasets_sd]
neg = [sign_dist[ds]["frac_negative"] for ds in datasets_sd]
zero = [sign_dist[ds]["frac_near_zero"] for ds in datasets_sd]

x = np.arange(len(datasets_sd))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, pos, label="Positive", color="#27AE60", edgecolor="white", linewidth=0.5)
ax.bar(x, neg, bottom=pos, label="Negative", color="#E74C3C", edgecolor="white", linewidth=0.5)
bottoms = [p + n for p, n in zip(pos, neg)]
ax.bar(x, zero, bottom=bottoms, label="Near-zero", color="#BDC3C7", edgecolor="white", linewidth=0.5)

ax.set_xlabel("Dataset", fontsize=11)
ax.set_ylabel("Fraction of CoI Pairs", fontsize=11)
ax.set_title("Figure 4: Co-Information Sign Distribution Across Real Datasets", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([DS_DISPLAY.get(d, d) for d in datasets_sd], rotation=30, ha="right")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

print("All 4 figures generated successfully.")

All 4 figures generated successfully.


## Results Summary

Display the main results table (8 methods x 8 datasets), key statistical test outcomes,
module recovery results on synthetic data, and aggregate metrics.

In [12]:
# ── Main Results Table ──
def _fmt(val, digits=3):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return "---"
    return f"{val:.{digits}f}"

print("=" * 80)
print("TABLE 1: Balanced Accuracy (mean +/- std) — 8 Methods x 8 Datasets")
print("=" * 80)
header = f"{'Dataset':<12}" + "".join(f"{METHOD_DISPLAY.get(m, m):>10}" for m in ALL_METHODS)
print(header)
print("-" * len(header))
for ds in ALL_REAL_DATASETS:
    row = f"{DS_DISPLAY.get(ds, ds):<12}"
    means = [table1.get(ds, {}).get(m, {}).get("mean") for m in ALL_METHODS]
    best_idx = -1
    best_val = -1
    for i, v in enumerate(means):
        if v is not None and v > best_val:
            best_val = v
            best_idx = i
    for i, m in enumerate(ALL_METHODS):
        v = means[i]
        s = _fmt(v) if v else "---"
        if i == best_idx:
            s = f"*{s}*"
        row += f"{s:>10}"
    print(row)

# ── Statistical Tests Summary ──
print("\n" + "=" * 80)
print("STATISTICAL TESTS SUMMARY")
print("=" * 80)
meta = data["metadata"]
tests = [
    ("Friedman chi2", f"{meta['friedman_chi2']:.2f}", f"p={meta['friedman_p']:.2e}"),
    ("Wilcoxon arity (US vs RO)", f"p={meta['wilcoxon_arity_p']:.4f}", f"Cohen's d={meta['wilcoxon_arity_cohens_d']:.3f}"),
    ("Hedges' g (US vs SS)", f"g={meta['signed_vs_unsigned_pooled_hedges_g']:.4f}", ""),
    ("Spearman frustration", f"rho={meta['frustration_spearman_rho']:.4f}", f"p={meta['frustration_spearman_p']:.4f}"),
]
for name, stat, extra in tests:
    print(f"  {name:<30} {stat:<20} {extra}")

# ── Per-Method Mean Balanced Accuracy ──
print("\n" + "=" * 80)
print("PER-METHOD MEAN BALANCED ACCURACY (across 7 clf datasets)")
print("=" * 80)
pmba = meta["per_method_mean_balanced_accuracy"]
sorted_methods = sorted(pmba.items(), key=lambda x: x[1], reverse=True)
for meth, val in sorted_methods:
    bar = "#" * int(val * 50)
    print(f"  {METHOD_DISPLAY.get(meth, meth):<10} {val:.4f} {bar}")

# ── Module Recovery (Synthetic) ──
print("\n" + "=" * 80)
print("TABLE 4: Module Recovery (Jaccard) on Synthetic Datasets")
print("=" * 80)
header = f"{'Variant':<25}" + "".join(f"{METHOD_DISPLAY[m]:>12}" for m in CLUSTERING_METHODS)
print(header)
print("-" * len(header))
for variant in SYNTHETIC_VARIANTS:
    row = f"{variant:<25}"
    for meth in CLUSTERING_METHODS:
        mr = module_recovery.get((variant, meth), {})
        j = mr.get("jaccard_mean")
        row += f"{_fmt(j):>12}" if j is not None else f"{'---':>12}"
    print(row)

# ── Per-Method Mean Arity ──
print("\n" + "=" * 80)
print("PER-METHOD MEAN ARITY (FIGS methods)")
print("=" * 80)
pma = meta["per_method_mean_arity"]
for meth in FIGS_METHODS:
    val = pma.get(meth, 0)
    bar = "#" * int(val * 2)
    print(f"  {METHOD_DISPLAY.get(meth, meth):<10} {val:.3f} {bar}")

print("\n" + "=" * 80)
print(f"Paper compiled: {'Yes' if meta.get('paper_compiled') else 'No'}")
print(f"Total examples: {sum(len(d['examples']) for d in data['datasets'])}")
print(f"Datasets: {len(data['datasets'])} (real={len(ALL_REAL_DATASETS)}, synthetic={len(SYNTHETIC_VARIANTS)})")
print("=" * 80)

TABLE 1: Balanced Accuracy (mean +/- std) — 8 Methods x 8 Datasets
Dataset        AA-FIGS   RO-FIGS   US-FIGS   SS-FIGS   HT-FIGS       EBM        RF    Linear
--------------------------------------------------------------------------------------------
Adult            0.678     0.708     0.703     0.702     0.703   *0.715*     0.706     0.672
Elec.            0.793     0.781     0.791     0.794     0.790     0.844   *0.884*     0.740
Jannis           0.727     0.734     0.734     0.741     0.742     0.772   *0.784*     0.729
Higgs            0.669     0.670     0.676     0.671     0.674     0.715   *0.719*     0.633
Eye              0.589     0.580     0.573     0.582     0.576   *0.641*     0.637     0.560
Credit           0.765     0.767     0.768     0.767     0.765   *0.775*     0.772     0.724
MiniBoo.         0.876     0.887     0.902     0.892     0.900     0.930   *0.933*     0.872
CalHous.           ---       ---       ---       ---       ---   *0.836*     0.810     0.601

ST